# RFP Field-Aware Generation Quickcheck

이 노트북은 P4 HWPX corpus의 retrieval 결과를 입력으로 받아 generation 단계를 빠르게 점검합니다.

핵심은 검색된 context를 그대로 LLM에 길게 넣는 것이 아니라, 입력 JSON 구조를 먼저 분석한 뒤 질문 유형별로 필요한 field를 골라 context를 구성하는 것입니다.

기본 흐름은 다음과 같습니다.

1. 상단 설정값 확인
2. 입력 JSONL 구조 분석
3. retrieval 결과와 chunk metadata 로드
4. 질문 유형 분류 및 field-aware context 구성
5. Qwen2.5-3B-Instruct로 JSON 답변 생성
6. deterministic 검증 및 결과 저장
7. 선택적으로 RAGAS 평가 실행

기본 공유 모드는 `USE_SOURCE_STORE=False`입니다. `source_store`는 임베딩 대상이 아니며, 추후 고도화 실험에서만 ID 기반 원문 확장 lookup 용도로 사용합니다.


## 0. 설치 안내

새 Colab 런타임에서 실행한다면 아래 셀의 주석을 해제해서 필요한 패키지를 설치하세요. 이미 설치되어 있으면 건너뛰면 됩니다.


In [ ]:
# RAGAS / LangChain 재현성 확보용 고정 설치
!pip uninstall -y ragas langchain langchain-core langchain-community langchain-text-splitters langchain-openai langsmith instructor

!pip install -q \
  "ragas==0.2.15" \
  "langchain==0.2.17" \
  "langchain-core==0.2.43" \
  "langchain-community==0.2.17" \
  "langchain-text-splitters==0.2.4" \
  "langchain-openai==0.1.25" \
  "datasets" \
  "openai"


In [1]:
import ragas
import langchain
import langchain_core
import langchain_community

print("ragas:", ragas.__version__)
print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)
print("langchain_community:", langchain_community.__version__)

ragas: 0.2.15
langchain: 0.2.17
langchain_core: 0.2.43
langchain_community: 0.2.17


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/DB_RAG_Codeit_Project")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("exists:", PROJECT_ROOT.exists())
print("SRC_DIR:", SRC_DIR)
print("src exists:", SRC_DIR.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/DB_RAG_Codeit_Project
exists: True
SRC_DIR: /content/drive/MyDrive/DB_RAG_Codeit_Project/src
src exists: True


## 1. 상단 설정

팀원이 가장 먼저 수정해야 하는 영역입니다. `CORPUS_N`, `PARSING_OUTPUT_NAME`, `RUN_NAME`, `RETRIEVAL_EXPERIMENT_ID`만 정확히 맞추면 같은 generation 흐름을 재사용할 수 있습니다.


In [2]:
from pathlib import Path
from datetime import datetime

# PROJECT_ROOT: 노트북이 어느 위치에서 실행되든 프로젝트 루트를 명확히 잡기 위한 경로입니다.
# Colab에서는 Google Drive에 올린 프로젝트 폴더를 사용하고, 로컬에서는 Windows 경로를 fallback으로 사용합니다.
PROJECT_ROOT = Path("/content/drive/MyDrive/DB_RAG_Codeit_Project")
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path(r"C:\Users\yoosy\Desktop\codeit\chatbot")

# CORPUS_N: 125, 250, 690 중 어떤 corpus를 사용할지 선택합니다.
CORPUS_N = 125

# PARSING_OUTPUT_NAME: chunk 파일이 들어 있는 parsing 산출물 폴더명입니다.
PARSING_OUTPUT_NAME = f"parsing_p4_hwpx_{CORPUS_N}_datafix_recallguard"

# RUN_NAME: retrieval 단계에서 저장한 generation 입력 폴더 이름입니다.
RUN_NAME = "exp100_0525_exp1_recallguard_docfill_metadatafilter"

# RETRIEVAL_EXPERIMENT_ID: 여러 retrieval 조건 중 generation 입력으로 사용할 조건입니다.
RETRIEVAL_EXPERIMENT_ID = "J5_hybrid_rrf_rerank"

# EXPERIMENT_NAME: 이번 generation 실험을 구분하기 위한 사람이 읽기 쉬운 이름입니다.
EXPERIMENT_NAME = f"generation_review50_{RETRIEVAL_EXPERIMENT_ID}"

# RUN_TIMESTAMP: 같은 실험을 반복해도 output이 덮어써지지 않도록 붙이는 실행 시간입니다.
RUN_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

# 경로 설정은 pathlib.Path로 관리합니다.
PARSE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / PARSING_OUTPUT_NAME
CHUNKS_PATH = PARSE_OUTPUT_DIR / f"chunks_v2_{CORPUS_N}.jsonl"
SOURCE_STORE_PATH = PARSE_OUTPUT_DIR / f"source_store_v2_{CORPUS_N}.jsonl"
RETRIEVAL_BASE_DIR = PROJECT_ROOT / "outputs"
RETRIEVAL_RUN_DIR = RETRIEVAL_BASE_DIR / RUN_NAME
OUTPUT_BASE_DIR = PROJECT_ROOT / "outputs" / f"generation_p4_hwpx_{CORPUS_N}"
OUTPUT_DIR = OUTPUT_BASE_DIR / f"{EXPERIMENT_NAME}_{RUN_TIMESTAMP}"

# 직접 파일을 지정해야 할 때만 Path를 넣습니다. None이면 RETRIEVAL_RUN_DIR에서 자동 탐색합니다.
INPUT_CONTEXTS_PATH = None
INPUT_RESULTS_PATH = None

# source_store는 기본적으로 사용하지 않습니다. 파일이 없으면 자동으로 False처럼 동작합니다.
USE_SOURCE_STORE = False

# review50: 30~50문항 수동 검수용입니다. exp100은 100문항 자동 요약용으로 바꾸면 됩니다.
GENERATION_SAMPLE_SIZE = 50
REVIEW_FOCUS = True
RANDOM_SEED = 42

# context와 모델 출력 길이를 제한해서 OOM 위험을 줄입니다.
MAX_CONTEXT_CHARS = 8000
MAX_NEW_TOKENS = 1024
GENERATION_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
FALLBACK_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# 기본은 greedy decoding입니다. Colab에서 hard error가 날 때만 sampling fallback을 켭니다.
GENERATION_DO_SAMPLE = False
GENERATION_TEMPERATURE = 0.1
GENERATION_TOP_P = 0.9
GENERATION_TOP_K = 20

# 디버깅만 할 때 True로 바꾸면 실제 LLM 호출 없이 저장 구조와 검증 흐름만 확인합니다.
USE_FAKE_GENERATOR_FOR_DEBUG = False

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CHUNKS_PATH:", CHUNKS_PATH, "exists=", CHUNKS_PATH.exists())
print("SOURCE_STORE_PATH:", SOURCE_STORE_PATH, "exists=", SOURCE_STORE_PATH.exists())
print("RETRIEVAL_RUN_DIR:", RETRIEVAL_RUN_DIR, "exists=", RETRIEVAL_RUN_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)
print("GENERATION_DO_SAMPLE:", GENERATION_DO_SAMPLE)
print("MAX_NEW_TOKENS:", MAX_NEW_TOKENS)


PROJECT_ROOT: /content/drive/MyDrive/DB_RAG_Codeit_Project
CHUNKS_PATH: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p4_hwpx_125_datafix_recallguard/chunks_v2_125.jsonl exists= True
SOURCE_STORE_PATH: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/parsing_p4_hwpx_125_datafix_recallguard/source_store_v2_125.jsonl exists= True
RETRIEVAL_RUN_DIR: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/exp100_0525_exp1_recallguard_docfill_metadatafilter exists= True
OUTPUT_DIR: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_022629
GENERATION_DO_SAMPLE: False
MAX_NEW_TOKENS: 1024


## 2. 모듈 import와 output 폴더 생성

`OUTPUT_DIR`는 timestamp가 포함된 새 폴더로 생성합니다. 같은 실험을 반복해도 기존 결과를 덮어쓰지 않습니다.


In [4]:
import importlib
import sys

sys.modules.pop("generation", None)
sys.modules.pop("generation.rfp_generation", None)
importlib.invalidate_caches()

from generation.rfp_generation import (
    build_context_package,
    build_prompt,
    build_ragas_eval_records,
    enrich_generation_record,
    inspect_jsonl_structure,
    load_chunk_index,
    load_source_store_index,
    postprocess_answer,
    prepare_generation_items,
    read_csv_records,
    read_jsonl,
    save_generation_outputs,
    summarize_ragas_scores,
    write_json,
    write_jsonl,
)

# 설정 셀에서 이미 OUTPUT_DIR 이름을 만들었으므로, 여기서는 덮어쓰기 방지를 위해 exist_ok=False로 생성합니다.
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print("created OUTPUT_DIR:", OUTPUT_DIR)


created OUTPUT_DIR: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047


## 3. Colab/GPU 메모리 정리

retrieval 노트북을 같은 런타임에서 먼저 실행했다면, embedding model, reranker, Chroma client 참조를 제거한 뒤 generation 모델을 로드하는 편이 안전합니다.


In [5]:
# retrieval 노트북과 같은 런타임에서 이어서 실행할 때만 의미가 있습니다.
# 존재하지 않는 이름은 건너뜁니다.
import gc

for name in ["embedding_model", "reranker", "chroma_client", "client", "collection", "vector_db"]:
    if name in globals():
        del globals()[name]

gc.collect()

try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("cuda memory cleared")
    else:
        print("cuda is not available")
except ImportError:
    print("torch is not installed yet")


cuda memory cleared


## 4. 입력 파일 자동 탐색

retrieval 결과는 `all_experiment_results_*.csv`와 `all_experiment_contexts_*.jsonl` 조합을 우선 사용합니다. 파일명이 다르면 상단 설정에서 직접 지정하세요.


In [6]:
def find_one_file(base_dir: Path, primary_patterns: list[str], fallback_patterns: list[str] | None = None) -> Path:
    if not base_dir.exists():
        parent = base_dir.parent
        candidates = sorted([p.name for p in parent.glob("*")]) if parent.exists() else []
        raise FileNotFoundError(
            f"retrieval run directory not found: {base_dir}\n"
            f"available under {parent}: {candidates[:30]}"
        )

    def collect(patterns: list[str]) -> list[Path]:
        matches = []
        for pattern in patterns:
            matches.extend(sorted(base_dir.glob(pattern)))
        return matches

    matches = collect(primary_patterns)
    if not matches and fallback_patterns:
        matches = collect(fallback_patterns)

    if not matches:
        existing = sorted([p.name for p in base_dir.glob("*")])
        raise FileNotFoundError(
            f"no file matched primary={primary_patterns}, fallback={fallback_patterns} under {base_dir}\n"
            f"existing files: {existing[:50]}"
        )
    return matches[-1]

if INPUT_CONTEXTS_PATH is None:
    INPUT_CONTEXTS_PATH = find_one_file(
        RETRIEVAL_RUN_DIR,
        [f"generation_predictions_{RETRIEVAL_EXPERIMENT_ID}_*.jsonl"],
        ["generation_predictions_*.jsonl"],
    )
else:
    INPUT_CONTEXTS_PATH = Path(INPUT_CONTEXTS_PATH)

if INPUT_RESULTS_PATH is None:
    INPUT_RESULTS_PATH = INPUT_CONTEXTS_PATH
else:
    INPUT_RESULTS_PATH = Path(INPUT_RESULTS_PATH)

print("INPUT_CONTEXTS_PATH:", INPUT_CONTEXTS_PATH)
print("INPUT_RESULTS_PATH:", INPUT_RESULTS_PATH)


INPUT_CONTEXTS_PATH: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/exp100_0525_exp1_recallguard_docfill_metadatafilter/generation_predictions_J5_hybrid_rrf_rerank_exp100_0525_exp1_recallguard_docfill_metadatafilter.jsonl
INPUT_RESULTS_PATH: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/exp100_0525_exp1_recallguard_docfill_metadatafilter/generation_predictions_J5_hybrid_rrf_rerank_exp100_0525_exp1_recallguard_docfill_metadatafilter.jsonl


## 5. 입력 JSON 구조 분석

generation을 시작하기 전에 실제 JSONL key 구조를 확인합니다. 이 결과를 기준으로 context builder가 어떤 field를 쓸 수 있는지 판단합니다.


In [7]:
chunk_schema = inspect_jsonl_structure(CHUNKS_PATH, sample_size=10)
print("[chunks_v2 schema]")
print("top-level keys:", chunk_schema["top_level_keys"])
print("canonical mapping:")
for field, info in chunk_schema["canonical_field_mapping"].items():
    if info.get("path"):
        print(f"  {field}: {info['path']} (present_ratio={info['present_ratio']})")

write_json(OUTPUT_DIR / "input_chunk_schema_summary.json", chunk_schema)
print("saved:", OUTPUT_DIR / "input_chunk_schema_summary.json")

if str(INPUT_CONTEXTS_PATH).endswith(".jsonl"):
    context_schema = inspect_jsonl_structure(INPUT_CONTEXTS_PATH, sample_size=10)
    print("\n[retrieval contexts schema]")
    print("top-level keys:", context_schema["top_level_keys"])
    write_json(OUTPUT_DIR / "input_context_schema_summary.json", context_schema)
else:
    print("\nINPUT_CONTEXTS_PATH is CSV. JSONL structure inspection is skipped for contexts.")


[chunks_v2 schema]
top-level keys: ['answer_allowed_question_types', 'answer_blocked_question_types', 'answer_policy', 'answer_risk_level', 'budget_answer_enabled', 'canonical_doc_id', 'canonical_doc_key', 'chunk_id', 'chunk_type', 'content', 'doc_id', 'doc_key', 'eligibility_answer_enabled', 'embed_enabled', 'evidence_id', 'evidence_text_short', 'fact_confidence', 'fact_status', 'fact_type', 'metadata', 'payment_answer_enabled', 'retrieval_role', 'source_file', 'source_file_nfc', 'source_format', 'source_ref']
canonical mapping:
  source_file: source_file (present_ratio=1.0)
  chunk_id: chunk_id (present_ratio=1.0)
  section_title: metadata.section_path (present_ratio=1.0)
  text: content (present_ratio=1.0)
  fact: fact_type (present_ratio=0.2)
  fact_candidates: content (present_ratio=1.0)
  metadata: metadata (present_ratio=1.0)
  source_store_id: source_ref.source_store_id (present_ratio=1.0)
saved: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/genera

## 6. retrieval 결과 로드와 review50 item 구성

`RETRIEVAL_EXPERIMENT_ID`에 해당하는 결과만 골라 generation 입력으로 사용합니다.


In [8]:
rfp_path = PROJECT_ROOT / "src" / "generation" / "rfp_generation.py"
text = rfp_path.read_text(encoding="utf-8")
print("load_generation_input_rows in file:", "load_generation_input_rows" in text)

load_generation_input_rows in file: True


In [9]:
import generation.rfp_generation as rfp_gen
load_generation_input_rows = rfp_gen.load_generation_input_rows

result_rows, context_rows = load_generation_input_rows(
    INPUT_RESULTS_PATH,
    INPUT_CONTEXTS_PATH,
    experiment_id=RETRIEVAL_EXPERIMENT_ID,
)

items = prepare_generation_items(
    result_rows,
    context_rows,
    experiment_id=RETRIEVAL_EXPERIMENT_ID,
    sample_size=GENERATION_SAMPLE_SIZE,
    review_focus=REVIEW_FOCUS,
    random_seed=RANDOM_SEED,
)

if not items:
    raise ValueError("No generation items were prepared. Check RETRIEVAL_EXPERIMENT_ID and input files.")

print("result rows:", len(result_rows))
print("context rows:", len(context_rows))
print("generation items:", len(items))
print("first question:", items[0]["question_id"], items[0]["question"][:120])

result rows: 100
context rows: 452
generation items: 50
first question: Q005 그랜드코리아레저(주)의 그룹웨어 구축 사업에서 목표로 하는 시스템 개선의 대상 범위를 모두 열거해 주십시오.


## 7. chunk metadata와 source_store 선택 로드

context builder는 retrieval row의 `chunk_id`를 이용해 `chunks_v2`에서 metadata를 조회합니다.

이번 수정에서는 검색된 chunk만 보지 않고, **검색된 문서(source_file)의 핵심 fact 후보**도 `chunks_v2`에서 함께 로드합니다. 이렇게 해야 Q008처럼 정답 문서는 검색됐지만 예산 fact chunk가 빠진 경우를 줄일 수 있습니다.

`source_store`는 기본적으로 로드하지 않습니다.


In [10]:
def _context_source_file(ctx):
    metadata = ctx.get("metadata") if isinstance(ctx.get("metadata"), dict) else {}
    return str(ctx.get("source_file") or ctx.get("filename") or metadata.get("source_file") or "")


chunk_ids = {
    str(ctx.get("chunk_id", ""))
    for item in items
    for ctx in item.get("retrieved_contexts", [])
    if ctx.get("chunk_id")
}
retrieved_source_files = {
    source_file
    for item in items
    for ctx in item.get("retrieved_contexts", [])
    for source_file in [_context_source_file(ctx)]
    if source_file
}
# 검색된 chunk 자체뿐 아니라, 검색된 문서의 핵심 fact 후보도 함께 로드합니다.
# source_store를 여는 것이 아니라 chunks_v2 안에서 같은 source_file의 fact_candidates만 보강 조회하는 방식입니다.
chunk_index = load_chunk_index(
    CHUNKS_PATH,
    chunk_ids=chunk_ids,
    source_files=retrieved_source_files,
    fact_types={
        "document_identity",
        "document_summary",
        "project_budget",
        "budget",
        "estimated_price",
        "base_amount",
        "project_duration",
        "submission_documents",
        "submission_logistics",
        "eligibility",
        "business_type",
    },
)
print("chunk ids from retrieval:", len(chunk_ids))
print("retrieved source files:", len(retrieved_source_files))
print("loaded chunk/fact metadata:", len(chunk_index))

source_store_ids = {
    str(chunk.get("source_ref", {}).get("source_store_id", ""))
    for chunk in chunk_index.values()
    if isinstance(chunk.get("source_ref"), dict) and chunk.get("source_ref", {}).get("source_store_id")
}

if USE_SOURCE_STORE and SOURCE_STORE_PATH.exists():
    source_store_index = load_source_store_index(SOURCE_STORE_PATH, source_store_ids, enabled=True)
else:
    source_store_index = {}
    USE_SOURCE_STORE = False

print("USE_SOURCE_STORE:", USE_SOURCE_STORE)
print("loaded source_store records:", len(source_store_index))


chunk ids from retrieval: 204
retrieved source files: 96
loaded chunk/fact metadata: 889
USE_SOURCE_STORE: False
loaded source_store records: 0


## 8. context builder quick preview

실제 LLM 호출 전에 질문 유형 분류와 context 조립 결과를 3개만 확인합니다.


In [11]:
context_config = {
    "max_context_chars_fact": MAX_CONTEXT_CHARS,
    "max_context_chars_synthesis": MAX_CONTEXT_CHARS,
    "max_blocks_fact": 8,
    "max_blocks_synthesis": 12,
    "evidence_text_chars": 900,
}

preview_packages = []
for item in items[:3]:
    package = build_context_package(
        item["question"],
        item.get("retrieved_contexts", []),
        chunk_index=chunk_index,
        source_store_index=source_store_index,
        use_source_store=USE_SOURCE_STORE,
        config=context_config,
    )
    preview_packages.append(package)
    print("=" * 80)
    print(item["question_id"], item["question"])
    print("analysis:", package["question_analysis"])
    print(package["context_text"][:1500])


Q005 그랜드코리아레저(주)의 그룹웨어 구축 사업에서 목표로 하는 시스템 개선의 대상 범위를 모두 열거해 주십시오.
analysis: {'question_types': ['business_type', 'requirements'], 'period_subtypes': [], 'is_multi_doc': False, 'is_multi_intent': True, 'needs_synthesis': True, 'answer_type': 'summary', 'intent_slots': ['purpose_summary', 'requirements_list'], 'intent_plan': [{'intent_id': 'I01', 'intent': 'purpose_summary', 'answer_section': '핵심 요약', 'targets': [], 'target_policy': 'context_scope', 'required_fact_types': ['document_summary', 'business_type', 'requirements'], 'preferred_chunk_types': ['text', 'table', 'fact_candidates'], 'requires_computation': False, 'requires_all_targets': False, 'classification_signals': ['목표']}, {'intent_id': 'I02', 'intent': 'requirements_list', 'answer_section': '목록', 'targets': [], 'target_policy': 'context_scope', 'required_fact_types': ['requirements', 'business_type'], 'preferred_chunk_types': ['table', 'text', 'fact_candidates'], 'requires_computation': False, 'requires_all_targets': False, 'c

## 9. LLM 로드

기본 모델은 `Qwen/Qwen2.5-3B-Instruct`입니다. OOM이 나면 상단 설정에서 sample/context/token을 줄이고, 마지막 fallback으로 1.5B 모델을 사용하세요.


In [12]:
def load_generation_model(model_name: str):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    print("loading tokenizer:", model_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:
        tokenizer.pad_token = tokenizer.eos_token

    print("loading model:", model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()
    return tokenizer, model


def generate_json_answer(messages, tokenizer, model, *, max_new_tokens: int) -> str:
    import torch

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": GENERATION_DO_SAMPLE,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if GENERATION_DO_SAMPLE:
        generation_kwargs.update(
            {
                "temperature": GENERATION_TEMPERATURE,
                "top_p": GENERATION_TOP_P,
                "top_k": GENERATION_TOP_K,
            }
        )

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            **generation_kwargs,
        )
    new_tokens = generated_ids[:, inputs.input_ids.shape[-1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()


def fake_json_answer(context_package):
    return json.dumps(
        {
            "answer": "디버그 모드입니다. 실제 LLM을 호출하지 않았습니다.",
            "answer_type": context_package.get("question_analysis", {}).get("answer_type", "unknown"),
            "confidence": "low",
            "is_answerable": False,
            "final_values": {},
            "documents": [],
            "missing_info": ["debug_generation"],
            "warnings": ["USE_FAKE_GENERATOR_FOR_DEBUG=True"],
        },
        ensure_ascii=False,
    )

if USE_FAKE_GENERATOR_FOR_DEBUG:
    tokenizer, model = None, None
    print("fake generator mode: model loading skipped")
else:
    tokenizer, model = load_generation_model(GENERATION_MODEL_NAME)


loading tokenizer: Qwen/Qwen2.5-3B-Instruct


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

loading model: Qwen/Qwen2.5-3B-Instruct


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 10. review50 generation 실행

각 질문에 대해 context를 구성하고, LLM 답변을 JSON schema로 후처리한 뒤 deterministic 검증 결과를 함께 저장합니다.


In [13]:
import time

records = []
context_packages = []

for idx, item in enumerate(items, start=1):
    context_package = build_context_package(
        item["question"],
        item.get("retrieved_contexts", []),
        chunk_index=chunk_index,
        source_store_index=source_store_index,
        use_source_store=USE_SOURCE_STORE,
        config=context_config,
    )
    messages = build_prompt(context_package)

    start = time.perf_counter()
    if USE_FAKE_GENERATOR_FOR_DEBUG:
        raw_text = fake_json_answer(context_package)
    else:
        raw_text = generate_json_answer(
            messages,
            tokenizer,
            model,
            max_new_tokens=MAX_NEW_TOKENS,
        )
    generation_ms = (time.perf_counter() - start) * 1000

    answer = postprocess_answer(raw_text, context_package)
    record = enrich_generation_record(
        answer,
        item,
        context_package,
        generation_ms=generation_ms,
        model_name=GENERATION_MODEL_NAME if not USE_FAKE_GENERATOR_FOR_DEBUG else "fake-generator",
        experiment_name=EXPERIMENT_NAME,
        run_timestamp=RUN_TIMESTAMP,
    )
    records.append(record)

    context_copy = dict(context_package)
    context_copy["question_id"] = item.get("question_id", "")
    context_packages.append(context_copy)

    if idx % 5 == 0 or idx == len(items):
        print(f"generated {idx}/{len(items)}")

print("done:", len(records))


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


generated 5/50
generated 10/50
generated 15/50
generated 20/50
generated 25/50
generated 30/50
generated 35/50
generated 40/50
generated 45/50
generated 50/50
done: 50


👉 50개 생성 기준 7m 13s -> 9m 35s 소요됨

## 11. 결과 저장

목표 문서에서 요구한 output 파일명을 기준으로 저장합니다.


In [15]:
RUN_RAGAS_EVAL = True

In [16]:
run_config = {
    "project_root": str(PROJECT_ROOT),
    "corpus_n": CORPUS_N,
    "parsing_output_name": PARSING_OUTPUT_NAME,
    "chunks_path": str(CHUNKS_PATH),
    "retrieval_run_name": RUN_NAME,
    "retrieval_experiment_id": RETRIEVAL_EXPERIMENT_ID,
    "input_contexts_path": str(INPUT_CONTEXTS_PATH),
    "input_results_path": str(INPUT_RESULTS_PATH),
    "experiment_name": EXPERIMENT_NAME,
    "run_timestamp": RUN_TIMESTAMP,
    "output_dir": str(OUTPUT_DIR),
    "use_source_store": USE_SOURCE_STORE,
    "generation_sample_size": GENERATION_SAMPLE_SIZE,
    "max_context_chars": MAX_CONTEXT_CHARS,
    "generation_model_name": GENERATION_MODEL_NAME if not USE_FAKE_GENERATOR_FOR_DEBUG else "fake-generator",
    "max_new_tokens": MAX_NEW_TOKENS,
    "generation_do_sample": GENERATION_DO_SAMPLE,
    "generation_temperature": GENERATION_TEMPERATURE if GENERATION_DO_SAMPLE else None,
    "generation_top_p": GENERATION_TOP_P if GENERATION_DO_SAMPLE else None,
    "generation_top_k": GENERATION_TOP_K if GENERATION_DO_SAMPLE else None,
    "run_ragas_eval": RUN_RAGAS_EVAL,
}

paths = save_generation_outputs(OUTPUT_DIR, records, run_config=run_config)
for name, path in paths.items():
    print(f"{name}: {path}")

# context package는 디버깅용으로 별도 저장합니다.
write_jsonl(OUTPUT_DIR / "generation_contexts.jsonl", context_packages)
print("generation_contexts:", OUTPUT_DIR / "generation_contexts.jsonl")


generated_answers: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/generated_answers.jsonl
review_samples: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/review_samples.csv
llm_answer_review: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/llm_answer_review.csv
llm_answer_review_html: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/llm_answer_review.html
metrics_summary: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/metrics_summary.json
failure_tags_summary: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf

## 12. deterministic metrics 확인

RAGAS를 붙이기 전에 JSON 형식, citation, 숫자 grounding, answerable 비율을 먼저 확인합니다.


In [17]:
import json
metrics = json.loads((OUTPUT_DIR / "metrics_summary.json").read_text(encoding="utf-8"))
print(json.dumps(metrics, ensure_ascii=False, indent=2))

failure_summary = json.loads((OUTPUT_DIR / "failure_tags_summary.json").read_text(encoding="utf-8"))
print("\nfailure tag counts:")
print(json.dumps(failure_summary.get("failure_tag_counts", {}), ensure_ascii=False, indent=2))


{
  "total_questions": 50,
  "total": 50,
  "valid_json_count": 48,
  "valid_json_rate": 0.96,
  "empty_answer_count": 0,
  "empty_answer_rate": 0.0,
  "answer_available_count": 50,
  "answer_available_rate": 1.0,
  "recovered_answer_count": 2,
  "recovered_answer_rate": 0.04,
  "parse_error_type_counts": {
    "valid_json": 48,
    "json_decode_error": 2
  },
  "citation_checked_count": 50,
  "citation_valid_rate": 0.94,
  "numeric_grounded_checked_count": 50,
  "numeric_grounded_rate": 0.86,
  "source_numeric_grounded_checked_count": 50,
  "source_numeric_grounded_rate": 0.86,
  "derived_numeric_valid_checked_count": 50,
  "derived_numeric_valid_rate": 1.0,
  "answerable_count": 40,
  "answerable_rate": 0.8,
  "generation_ms_avg": 11488.007943260012,
  "failure_tag_counts": {
    "multi_intent_incomplete": 15,
    "source_numeric_missing": 14,
    "citation_wrong_target": 2,
    "llm_hallucination_risk": 7,
    "target_doc_coverage_missing": 10,
    "llm_invalid_json": 2,
    "insuff

## 13. 사람이 검수할 review 샘플 확인

`review_samples.csv`를 열어 질문, 예측 answer_type, 선택된 source/chunk, 답변, citation, warnings를 확인합니다.


In [18]:
import pandas as pd
review_df = pd.read_csv(OUTPUT_DIR / "review_samples.csv")
review_df[[
    "question_id",
    "predicted_answer_type",
    "confidence",
    "is_answerable",
    "failure_tags",
    "answer",
]].head(10)


,question_id,predicted_answer_type,confidence,is_answerable,failure_tags,answer
0,Q005,summary,high,True,"[""multi_intent_incomplete""]",목표로 하는 시스템 개선의 대상 범위는 다음과 같습니다: 사내 SNS 및 업무용 메...
1,Q008,budget,low,False,"[""source_numeric_missing""]","한국수자원공사 용인 첨단 시스템반도체 국가산단 용수공급사업, 용인 첨단 시스템반도체..."
2,Q021,budget,low,False,"[""source_numeric_missing"", ""citation_wrong_tar...",우즈벡-키르기즈스탄 기후변화대응 스마트 관개시스템 구축사업의 사업예산 근거를 con...
3,Q039,budget,high,True,"[""llm_hallucination_risk"", ""source_numeric_mis...",그렌드코리아레져(주)가 2024년도에 츄진하는 그룹웨어 시스템 구축 용역의 전체 예...
4,Q041,summary,high,True,[],전사 포털 및 사용자 맞춤형 업무 포털 부문이 신규 도입 S/W 중 포함되어 있습니다.
5,Q045,summary,high,True,"[""llm_hallucination_risk"", ""source_numeric_mis...",그랜드코리아레저(주)의 그룹웨어 도입 사업의 주요 추진 배경 세 가지는 시스템 노후...
6,Q057,summary,medium,True,"[""multi_intent_incomplete""]",한국수자원공사 용인 첨단 시스템반도체 국가산단 용수공급사업 타당성조사 입찰 제안서류...
7,Q064,summary,high,True,[],사업목적 중 하나인 생산공급 시스템의 재구축 및 전환 방향은 SAP ERP Conv...
8,Q077,summary,medium,True,"[""multi_intent_incomplete""]",매월 10대 이상 별도 클라우드 노트북 PC를 무상으로 지급받을 수 있는 조건은 명...
9,Q084,summary,high,True,[],우즈베키스탄과 키르기즈스탄에서 구축하려는 사업의 성과 목표는 다음과 같습니다:\n1...


## 14. RAGAS 평가 입력 확인

RAGAS는 generation 답변을 보정하는 용도가 아니라, 생성 품질을 별도로 평가하는 용도입니다.


In [13]:
from pathlib import Path
import json


def inspect_ragas_file(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    empty_answers = sum(1 for row in rows if not row.get("answer"))
    empty_contexts = sum(1 for row in rows if not row.get("contexts"))
    return {
        "path": path,
        "rows": len(rows),
        "empty_answers": empty_answers,
        "empty_contexts": empty_contexts,
        "mtime": path.stat().st_mtime,
    }


# Preferred latest result. If this folder is not available in the current runtime,
# choose the best available RAGAS input under PROJECT_ROOT/outputs.
RAGAS_OUTPUT_DIR_OVERRIDE = PROJECT_ROOT / "outputs" / "generation_review50_J5_hybrid_rrf_rerank_20260525_055114"

candidate_files = []
if RAGAS_OUTPUT_DIR_OVERRIDE.exists():
    candidate_files.append(RAGAS_OUTPUT_DIR_OVERRIDE / "ragas_eval_input.jsonl")

current_file = OUTPUT_DIR / "ragas_eval_input.jsonl"
if current_file.exists():
    candidate_files.append(current_file)

outputs_dir = PROJECT_ROOT / "outputs"
if outputs_dir.exists():
    candidate_files.extend(outputs_dir.rglob("ragas_eval_input.jsonl"))

candidate_files = sorted(set(p for p in candidate_files if p.exists()))
if not candidate_files:
    raise FileNotFoundError(f"No ragas_eval_input.jsonl found under {outputs_dir}")

candidate_infos = [inspect_ragas_file(path) for path in candidate_files]
candidate_infos = sorted(
    candidate_infos,
    key=lambda x: (x["empty_answers"], -x["mtime"]),
)

print("RAGAS input candidates:")
for info in candidate_infos[:10]:
    print(
        f"- {info['path']} | rows={info['rows']} "
        f"empty_answer={info['empty_answers']} empty_context={info['empty_contexts']}"
    )

selected_info = candidate_infos[0]
ragas_eval_input_path = selected_info["path"]
RAGAS_OUTPUT_DIR = ragas_eval_input_path.parent

ragas_eval_records = []
with ragas_eval_input_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            ragas_eval_records.append(json.loads(line))

answer_rows = sum(1 for row in ragas_eval_records if row.get("answer"))
context_rows = sum(1 for row in ragas_eval_records if row.get("contexts"))
empty_answer_ids = [
    row.get("question_id")
    for row in ragas_eval_records
    if not row.get("answer")
]

print("SELECTED_RAGAS_OUTPUT_DIR:", RAGAS_OUTPUT_DIR)
print("selected ragas input path:", ragas_eval_input_path)
print("loaded ragas rows:", len(ragas_eval_records))
print("rows with answer:", answer_rows)
print("rows with contexts:", context_rows)
print("empty answer rows:", len(empty_answer_ids))
print("empty answer ids:", empty_answer_ids[:30])
print("sample keys:", list(ragas_eval_records[0].keys()) if ragas_eval_records else None)

if ragas_eval_records:
    sample = ragas_eval_records[0]
    preview = {
        "question_id": sample.get("question_id"),
        "question": sample.get("question"),
        "answer": sample.get("answer"),
        "ground_truth": sample.get("ground_truth"),
        "contexts_count": len(sample.get("contexts", [])),
    }
    print(json.dumps(preview, ensure_ascii=False, indent=2)[:2000])


RAGAS input candidates:
- /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/ragas_eval_input.jsonl | rows=50 empty_answer=0 empty_context=0
- /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_055114/ragas_eval_input.jsonl | rows=50 empty_answer=0 empty_context=0
- /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009/ragas_eval_input.jsonl | rows=50 empty_answer=26 empty_context=0
SELECTED_RAGAS_OUTPUT_DIR: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047
selected ragas input path: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/ragas_eval_input.jsonl
loaded ragas rows: 50
rows with ans

## 15. 선택 실행: RAGAS 평가

`RUN_RAGAS_EVAL=True`일 때만 실행합니다. RAGAS는 evaluator LLM/embedding 설정이 필요할 수 있습니다. 설치된 RAGAS 버전의 API가 다르면 이 섹션만 해당 버전에 맞춰 조정하세요.


In [20]:
!pip install -q python-dotenv

In [14]:
# RAGAS runtime settings.
# Keep this cell independent: it defines all RAGAS flags used below.
from dotenv import load_dotenv, dotenv_values
from pathlib import Path
import os

RUN_RAGAS_EVAL = True

# RAGAS_EVAL_SAMPLE_SIZE: RAGAS 평가에 사용할 샘플 수 (너무 많으면 시간 초과 또는 과도한 비용 발생할 수 있음)
RAGAS_EVAL_SAMPLE_SIZE = 10

ENV_PATH = PROJECT_ROOT / "notebooks" / "rag" / ".env"
print("RUN_RAGAS_EVAL:", RUN_RAGAS_EVAL)
print("RAGAS_EVAL_SAMPLE_SIZE:", RAGAS_EVAL_SAMPLE_SIZE)
print("env exists:", ENV_PATH.exists())
print("env path:", ENV_PATH)

if RUN_RAGAS_EVAL:
    if not ENV_PATH.exists():
        raise FileNotFoundError(f".env file not found: {ENV_PATH}")

    env_values = dotenv_values(ENV_PATH)
    print("env keys:", list(env_values.keys()))

    load_dotenv(dotenv_path=ENV_PATH, override=True)

    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    RAGAS_OPENAI_MODEL = os.getenv("RAGAS_OPENAI_MODEL", "gpt-4o-mini")

    if not OPENAI_API_KEY:
        raise ValueError("OPENAI_API_KEY is missing. Check notebooks/rag/.env")

    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("RAGAS_OPENAI_MODEL:", RAGAS_OPENAI_MODEL)
else:
    RAGAS_OPENAI_MODEL = os.getenv("RAGAS_OPENAI_MODEL", "gpt-4o-mini")
    print("RAGAS evaluation skipped by RUN_RAGAS_EVAL=False")


RUN_RAGAS_EVAL: True
RAGAS_EVAL_SAMPLE_SIZE: 10
env exists: True
env path: /content/drive/MyDrive/DB_RAG_Codeit_Project/notebooks/rag/.env
env keys: ['OPENAI_API_KEY', 'RAGAS_OPENAI_MODEL']
RAGAS_OPENAI_MODEL: gpt-5-nano


In [15]:
# 15. Optional RAGAS evaluation
import json
import warnings

from generation.rfp_generation import summarize_ragas_scores, write_json

warnings.filterwarnings(
    "ignore",
    message=".*google.generativeai.*",
    category=FutureWarning,
)


def load_ragas_eval_input(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


if "RAGAS_OUTPUT_DIR" not in globals():
    RAGAS_OUTPUT_DIR = OUTPUT_DIR
if "RUN_RAGAS_EVAL" not in globals():
    RUN_RAGAS_EVAL = False
if "RAGAS_EVAL_SAMPLE_SIZE" not in globals():
    RAGAS_EVAL_SAMPLE_SIZE = 3  

if "RAGAS_OPENAI_MODEL" not in globals():
    RAGAS_OPENAI_MODEL = "gpt-4o-mini"
if "ragas_eval_records" not in globals():
    ragas_eval_input_path = RAGAS_OUTPUT_DIR / "ragas_eval_input.jsonl"
    if not ragas_eval_input_path.exists():
        raise FileNotFoundError(f"ragas_eval_input.jsonl not found: {ragas_eval_input_path}")
    ragas_eval_records = load_ragas_eval_input(ragas_eval_input_path)

if RUN_RAGAS_EVAL:
    import ragas
    from datasets import Dataset
    from langchain_openai import ChatOpenAI
    from ragas import evaluate
    from ragas.metrics import (
        answer_relevancy,
        context_precision,
        context_recall,
        faithfulness,
    )

    class RagasSafeChatOpenAI(ChatOpenAI):
        """Drop sampling kwargs that some judge models do not accept."""

        def _get_request_payload(self, input_, *, stop=None, **kwargs):
            kwargs.pop("temperature", None)
            payload = super()._get_request_payload(input_, stop=stop, **kwargs)
            payload.pop("temperature", None)
            return payload

    print("ragas version:", getattr(ragas, "__version__", "unknown"))
    print("ragas judge model:", RAGAS_OPENAI_MODEL)

    ragas_eval_records_for_eval = [
        row for row in ragas_eval_records
        if row.get("answer") and row.get("contexts")
    ]
    skipped_empty_answer = [
        row.get("question_id")
        for row in ragas_eval_records
        if not row.get("answer")
    ]
    skipped_empty_context = [
        row.get("question_id")
        for row in ragas_eval_records
        if row.get("answer") and not row.get("contexts")
    ]

    if RAGAS_EVAL_SAMPLE_SIZE:
        ragas_eval_records_for_eval = ragas_eval_records_for_eval[:RAGAS_EVAL_SAMPLE_SIZE]

    print("RAGAS input rows:", len(ragas_eval_records))
    print("RAGAS eval rows:", len(ragas_eval_records_for_eval))
    print("RAGAS eval sample size:", RAGAS_EVAL_SAMPLE_SIZE)
    print("skipped empty-answer rows:", len(skipped_empty_answer))
    print("skipped empty-context rows:", len(skipped_empty_context))
    print("skipped empty-answer ids:", skipped_empty_answer[:30])

    if not ragas_eval_records_for_eval:
        raise ValueError("No RAGAS rows to evaluate. All answers or contexts are empty.")

    has_ground_truth = any(
        row.get("ground_truth") for row in ragas_eval_records_for_eval
    )
    data = {
        "question": [row["question"] for row in ragas_eval_records_for_eval],
        "answer": [row["answer"] for row in ragas_eval_records_for_eval],
        "contexts": [row["contexts"] for row in ragas_eval_records_for_eval],
    }

    metrics = [faithfulness, answer_relevancy, context_precision]
    if has_ground_truth:
        data["ground_truth"] = [
            row.get("ground_truth", "")
            for row in ragas_eval_records_for_eval
        ]
        metrics.append(context_recall)

    print("has_ground_truth:", has_ground_truth)
    print("metrics:", [getattr(metric, "name", str(metric)) for metric in metrics])

    dataset = Dataset.from_dict(data)
    ragas_llm = RagasSafeChatOpenAI(model=RAGAS_OPENAI_MODEL)

    ragas_per_question_path = RAGAS_OUTPUT_DIR / "ragas_per_question.csv"
    ragas_summary_path = RAGAS_OUTPUT_DIR / "ragas_metrics_summary.json"

    try:
        ragas_result = evaluate(
            dataset,
            metrics=metrics,
            llm=ragas_llm,
            raise_exceptions=False,
        )
        ragas_df = ragas_result.to_pandas()
        ragas_status = "completed"
        ragas_error = None
    except Exception as exc:
        ragas_df = None
        ragas_status = "failed"
        ragas_error = {
            "error_type": type(exc).__name__,
            "error_message": str(exc)[:2000],
        }

    if ragas_df is not None:
        ragas_df["question_id"] = [
            row["question_id"] for row in ragas_eval_records_for_eval
        ]
        ragas_df["answer_type"] = [
            row.get("answer_type", "") for row in ragas_eval_records_for_eval
        ]
        ragas_df["ground_truth"] = [
            row.get("ground_truth", "")
            for row in ragas_eval_records_for_eval
        ]
        ragas_df["source_files"] = [
            json.dumps(row.get("source_files", []), ensure_ascii=False)
            for row in ragas_eval_records_for_eval
        ]
        ragas_df["chunk_ids"] = [
            json.dumps(row.get("chunk_ids", []), ensure_ascii=False)
            for row in ragas_eval_records_for_eval
        ]
        ragas_df.to_csv(ragas_per_question_path, index=False, encoding="utf-8-sig")

        ragas_rows = ragas_df.to_dict("records")
        ragas_summary = summarize_ragas_scores(ragas_rows)
        ragas_summary["ragas_status"] = ragas_status
    else:
        ragas_summary = {"ragas_status": ragas_status}

    ragas_summary["judge_model"] = RAGAS_OPENAI_MODEL
    ragas_summary["evaluated_rows"] = len(ragas_eval_records_for_eval)
    ragas_summary["total_input_rows"] = len(ragas_eval_records)
    ragas_summary["skipped_empty_answer_rows"] = len(skipped_empty_answer)
    ragas_summary["skipped_empty_context_rows"] = len(skipped_empty_context)
    ragas_summary["skipped_empty_answer_question_ids"] = skipped_empty_answer[:50]
    if ragas_error:
        ragas_summary["ragas_error"] = ragas_error

    write_json(ragas_summary_path, ragas_summary)

    print("saved summary:", ragas_summary_path)
    if ragas_df is not None:
        print("saved per-question:", ragas_per_question_path)
        display(ragas_df.head())
    else:
        print("RAGAS evaluation failed. Summary file was saved for debugging.")
    print(json.dumps(ragas_summary, ensure_ascii=False, indent=2))
else:
    print("RUN_RAGAS_EVAL=False: RAGAS evaluation was skipped.")
    print("RAGAS input file:", RAGAS_OUTPUT_DIR / "ragas_eval_input.jsonl")


ragas version: 0.2.15
ragas judge model: gpt-5-nano
RAGAS input rows: 50
RAGAS eval rows: 10
RAGAS eval sample size: 10
skipped empty-answer rows: 0
skipped empty-context rows: 0
skipped empty-answer ids: []
has_ground_truth: True
metrics: ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

saved summary: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/ragas_metrics_summary.json
saved per-question: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/ragas_per_question.csv


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall,question_id,answer_type,ground_truth,source_files,chunk_ids
0,그랜드코리아레저(주)의 그룹웨어 구축 사업에서 목표로 하는 시스템 개선의 대상 범위...,[[문서: 그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hw...,목표로 하는 시스템 개선의 대상 범위는 다음과 같습니다: 사내 SNS 및 업무용 메...,"사업 범위는 그룹웨어 시스템 구축, 기록물관리 시스템 구축, 업무용 메신저 시스템 ...",0.875,0.904198,1.000000,1.0,Q005,summary,"사업 범위는 그룹웨어 시스템 구축, 기록물관리 시스템 구축, 업무용 메신저 시스템 ...","[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp""]","[""doc_fcb9cb7e2d84_text_text_0002_part_001_758..."
1,한국수자원공사의 '용인 첨단 시스템반도체 국가산단 용수공급사업'과 한국수출입은행의 ...,[[문서: 한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp...,"한국수자원공사 용인 첨단 시스템반도체 국가산단 용수공급사업, 용인 첨단 시스템반도체...","한국수자원공사 용역 예산(2,392,940,000원)과 수출입은행 용역 예산(1,2...",0.000,0.000000,0.173611,0.0,Q008,budget,"한국수자원공사 용역 예산(2,392,940,000원)과 수출입은행 용역 예산(1,2...","[""한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp"", ""...","[""doc_6a4eb959610e_fact_candidates_fact_0002_p..."
2,사단법인아시아물위원회사무국에서 발주한 '우즈벡-키르기즈스탄 기후변화대응 스마트 관개...,[[문서: 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp |...,우즈벡-키르기즈스탄 기후변화대응 스마트 관개시스템 구축사업의 사업예산 근거를 con...,"해당 사업의 예산은 5,031,000,000원입니다.",0.000,0.894015,0.500000,1.0,Q021,budget,"해당 사업의 예산은 5,031,000,000원입니다.","[""사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp"", ""경기...","[""doc_5b75765779be_fact_candidates_fact_0002_p..."
3,그렌드코리아레져(쥬)가 이번 연도에 츄진하는 2024년도 구룹웨에 씨스탬 구쭉 용역...,[| cols: 3] 컬럼: Ⅰ | col_2 | 사업개요 1. 사업일반 □ 사 업...,그렌드코리아레져(주)가 2024년도에 츄진하는 그룹웨어 시스템 구축 용역의 전체 예...,그랜드코리아레저(주)가 추진을 기획하고 있는 '2024년도 GKL 그룹웨어 시스템 ...,0.000,0.925897,0.866667,1.0,Q039,budget,그랜드코리아레저(주)가 추진을 기획하고 있는 '2024년도 GKL 그룹웨어 시스템 ...,"[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp"",...","[""doc_fcb9cb7e2d84_fact_candidates_fact_0003_p..."
4,한국가스공사의 차세대 통합정보시스템(ERP) 구축 사업의 신규 도입 S/W 중 전사...,[개선 요구사항 도출 * 과업지시서 기능요구사항(SFR-012∼045) 34개 과제...,전사 포털 및 사용자 맞춤형 업무 포털 부문이 신규 도입 S/W 중 포함되어 있습니다.,"네, 해당 사업의 과업내용 중 주요 사업내용으로 본사 전사 포털 및 업무 포털의 신...",1.000,0.916863,0.750000,1.0,Q041,summary,"네, 해당 사업의 과업내용 중 주요 사업내용으로 본사 전사 포털 및 업무 포털의 신...","[""한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"", ""고려대학교...","[""doc_f1c6cddebe51_text_text_0004_part_002_e2f..."


{
  "faithfulness_mean": 0.3458333333333333,
  "faithfulness_low_questions": [
    "Q008",
    "Q021",
    "Q039",
    "Q045",
    "Q057",
    "Q077"
  ],
  "answer_relevancy_mean": 0.731874853447976,
  "answer_relevancy_low_questions": [
    "Q008",
    "Q077"
  ],
  "context_precision_mean": 0.584633838346166,
  "context_precision_low_questions": [
    "Q008",
    "Q021",
    "Q045",
    "Q064"
  ],
  "context_recall_mean": 0.4,
  "context_recall_low_questions": [
    "Q008",
    "Q045",
    "Q057",
    "Q064",
    "Q077",
    "Q084"
  ],
  "ragas_status": "completed",
  "judge_model": "gpt-5-nano",
  "evaluated_rows": 10,
  "total_input_rows": 50,
  "skipped_empty_answer_rows": 0,
  "skipped_empty_context_rows": 0,
  "skipped_empty_answer_question_ids": []
}


In [16]:
import pandas as pd
import json

if "RAGAS_OUTPUT_DIR" not in globals():
    RAGAS_OUTPUT_DIR = OUTPUT_DIR

print("RAGAS_OUTPUT_DIR:", RAGAS_OUTPUT_DIR)
print("summary:", RAGAS_OUTPUT_DIR / "ragas_metrics_summary.json")
print("per_question:", RAGAS_OUTPUT_DIR / "ragas_per_question.csv")

ragas_path = RAGAS_OUTPUT_DIR / "ragas_per_question.csv"
if ragas_path.exists():
    ragas_df = pd.read_csv(ragas_path)
    print("shape:", ragas_df.shape)
    print("columns:", ragas_df.columns.tolist())
    display(ragas_df.head(5))

    metric_cols = [
        "faithfulness",
        "answer_relevancy",
        "context_precision",
        "context_recall",
    ]
    for col in metric_cols:
        if col in ragas_df.columns:
            print(col, "non-null:", ragas_df[col].notna().sum(), "/", len(ragas_df))
else:
    print("RAGAS per-question file does not exist. RUN_RAGAS_EVAL=False 이거나 RAGAS를 아직 실행하지 않은 상태입니다.")


RAGAS_OUTPUT_DIR: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047
summary: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/ragas_metrics_summary.json
per_question: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/ragas_per_question.csv
shape: (10, 13)
columns: ['user_input', 'retrieved_contexts', 'response', 'reference', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall', 'question_id', 'answer_type', 'ground_truth', 'source_files', 'chunk_ids']


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall,question_id,answer_type,ground_truth,source_files,chunk_ids
0,그랜드코리아레저(주)의 그룹웨어 구축 사업에서 목표로 하는 시스템 개선의 대상 범위...,['[문서: 그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.h...,목표로 하는 시스템 개선의 대상 범위는 다음과 같습니다: 사내 SNS 및 업무용 메...,"사업 범위는 그룹웨어 시스템 구축, 기록물관리 시스템 구축, 업무용 메신저 시스템 ...",0.875,0.904198,1.000000,1.0,Q005,summary,"사업 범위는 그룹웨어 시스템 구축, 기록물관리 시스템 구축, 업무용 메신저 시스템 ...","[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp""]","[""doc_fcb9cb7e2d84_text_text_0002_part_001_758..."
1,한국수자원공사의 '용인 첨단 시스템반도체 국가산단 용수공급사업'과 한국수출입은행의 ...,['[문서: 한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hw...,"한국수자원공사 용인 첨단 시스템반도체 국가산단 용수공급사업, 용인 첨단 시스템반도체...","한국수자원공사 용역 예산(2,392,940,000원)과 수출입은행 용역 예산(1,2...",0.000,0.000000,0.173611,0.0,Q008,budget,"한국수자원공사 용역 예산(2,392,940,000원)과 수출입은행 용역 예산(1,2...","[""한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp"", ""...","[""doc_6a4eb959610e_fact_candidates_fact_0002_p..."
2,사단법인아시아물위원회사무국에서 발주한 '우즈벡-키르기즈스탄 기후변화대응 스마트 관개...,['[문서: 사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp ...,우즈벡-키르기즈스탄 기후변화대응 스마트 관개시스템 구축사업의 사업예산 근거를 con...,"해당 사업의 예산은 5,031,000,000원입니다.",0.000,0.894015,0.500000,1.0,Q021,budget,"해당 사업의 예산은 5,031,000,000원입니다.","[""사단법인아시아물위원회사무국_우즈벡-키르기즈스탄 기후변화대응 스.hwp"", ""경기...","[""doc_5b75765779be_fact_candidates_fact_0002_p..."
3,그렌드코리아레져(쥬)가 이번 연도에 츄진하는 2024년도 구룹웨에 씨스탬 구쭉 용역...,['| cols: 3] 컬럼: Ⅰ | col_2 | 사업개요 1. 사업일반 □ 사 ...,그렌드코리아레져(주)가 2024년도에 츄진하는 그룹웨어 시스템 구축 용역의 전체 예...,그랜드코리아레저(주)가 추진을 기획하고 있는 '2024년도 GKL 그룹웨어 시스템 ...,0.000,0.925897,0.866667,1.0,Q039,budget,그랜드코리아레저(주)가 추진을 기획하고 있는 '2024년도 GKL 그룹웨어 시스템 ...,"[""그랜드코리아레저(주)_2024년도 GKL 그룹웨어 시스템 구축 용역.hwp"",...","[""doc_fcb9cb7e2d84_fact_candidates_fact_0003_p..."
4,한국가스공사의 차세대 통합정보시스템(ERP) 구축 사업의 신규 도입 S/W 중 전사...,['개선 요구사항 도출 * 과업지시서 기능요구사항(SFR-012∼045) 34개 과...,전사 포털 및 사용자 맞춤형 업무 포털 부문이 신규 도입 S/W 중 포함되어 있습니다.,"네, 해당 사업의 과업내용 중 주요 사업내용으로 본사 전사 포털 및 업무 포털의 신...",1.000,0.916863,0.750000,1.0,Q041,summary,"네, 해당 사업의 과업내용 중 주요 사업내용으로 본사 전사 포털 및 업무 포털의 신...","[""한국가스공사_[재공고]차세대 통합정보시스템(ERP) 구축.hwp"", ""고려대학교...","[""doc_f1c6cddebe51_text_text_0004_part_002_e2f..."


faithfulness non-null: 10 / 10
answer_relevancy non-null: 10 / 10
context_precision non-null: 10 / 10
context_recall non-null: 10 / 10


## 16. 최종 산출물 경로 확인


In [22]:
print("OUTPUT_DIR:", OUTPUT_DIR)
for name, path in paths.items():
    print(f"{name}: {path}")
print("generation_contexts:", OUTPUT_DIR / "generation_contexts.jsonl")


OUTPUT_DIR: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009
generated_answers: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/generated_answers.jsonl
review_samples: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/review_samples.csv
llm_answer_review: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/llm_answer_review.csv
llm_answer_review_html: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/llm_answer_review.html
metrics_summary: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260526_021047/metrics

In [32]:
print("OUTPUT_DIR:", OUTPUT_DIR)

for name in [
    "generated_answers.jsonl",
    "review_samples.csv",
    "llm_answer_review.csv",
    "llm_answer_review.html",
    "generation_contexts.jsonl",
    "metrics_summary.json",
    "failure_tags_summary.json",
    "ragas_per_question.csv",
    "ragas_metrics_summary.json",
]:
    path = OUTPUT_DIR / name
    print(name, path.exists(), path)


OUTPUT_DIR: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009
generated_answers.jsonl True /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009/generated_answers.jsonl
review_samples.csv True /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009/review_samples.csv
llm_answer_review.csv False /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009/llm_answer_review.csv
llm_answer_review.html False /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009/llm_answer_review.html
generation_contexts.jsonl True /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review5

In [23]:
import json
from collections import Counter
import pandas as pd

analysis_rows = []
for r in records:
    analysis_rows.append({
        "question_id": r.get("question_id"),
        "answer_empty": not bool(str(r.get("answer") or "").strip()),
        "valid_json": bool(r.get("_valid_json")),
        "recovered_answer": bool(r.get("_recovered_answer")),
        "parse_error_type": r.get("_parse_error_type", ""),
        "is_answerable": r.get("is_answerable"),
        "failure_tags": json.dumps(r.get("_failure_tags", []), ensure_ascii=False),
        "warnings": json.dumps(r.get("warnings", []), ensure_ascii=False),
        "missing_info": json.dumps(r.get("missing_info", []), ensure_ascii=False),
        "raw_text_head": str(r.get("_raw_text") or "")[:600],
    })

analysis_df = pd.DataFrame(analysis_rows)
analysis_path = OUTPUT_DIR / "generation_parse_failure_analysis.csv"
analysis_df.to_csv(analysis_path, index=False, encoding="utf-8-sig")

print("saved:", analysis_path)
print("total:", len(analysis_df))
print("empty answers:", int(analysis_df["answer_empty"].sum()))
print("valid json:", int(analysis_df["valid_json"].sum()))
print("recovered answers:", int(analysis_df["recovered_answer"].sum()))
print("parse error counts:", Counter(analysis_df["parse_error_type"]))

display(
    analysis_df[
        analysis_df["answer_empty"] | analysis_df["recovered_answer"] | (~analysis_df["valid_json"])
    ].head(3)
)


saved: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009/generation_parse_failure_analysis.csv
total: 50
empty answers: 26
valid json: 24
recovered answers: 0
parse error counts: Counter({'': 50})


,question_id,answer_empty,valid_json,recovered_answer,parse_error_type,is_answerable,failure_tags,warnings,missing_info,raw_text_head
0,Q005,True,False,False,,False,"[""llm_invalid_json""]","[""LLM output was not valid JSON.""]","[""valid_json""]","{\n ""answer"": ""시스템 개선의 대상 범위는 다음과 같다: 임직원의 업무..."
1,Q008,True,False,False,,False,"[""llm_invalid_json""]","[""LLM output was not valid JSON.""]","[""valid_json""]","{\n ""answer"": ""두 사업의 예산 차액은 100,000,000원입니다.""..."
2,Q021,True,False,False,,False,"[""llm_invalid_json""]","[""LLM output was not valid JSON.""]","[""valid_json""]","{\n ""answer"": ""사단법인아시아물위원회사무국에서 발주한 '우즈벡-키르기즈..."


## 17. LLM 답변 형성 과정 검토 파일

`generated_answers.jsonl`은 정보가 많아 사람이 읽기 어렵습니다. 아래 셀은 GT와 함께 `Raw LLM Text → Parsed Answer → Final Answer`를 나란히 볼 수 있는 CSV/HTML을 다시 생성합니다.


In [24]:
from generation.rfp_generation import build_llm_answer_review_rows, write_llm_answer_review_html

llm_answer_review_rows = build_llm_answer_review_rows(records)
llm_answer_review_csv = OUTPUT_DIR / "llm_answer_review.csv"
llm_answer_review_html = OUTPUT_DIR / "llm_answer_review.html"

pd.DataFrame(llm_answer_review_rows).to_csv(
    llm_answer_review_csv,
    index=False,
    encoding="utf-8-sig",
)
write_llm_answer_review_html(llm_answer_review_html, llm_answer_review_rows)

print("saved csv:", llm_answer_review_csv)
print("saved html:", llm_answer_review_html)
display(pd.DataFrame(llm_answer_review_rows)[[
    "question_id",
    "question",
    "ground_truth",
    "parsed_answer",
    "final_answer",
    "valid_json",
    "recovered_answer",
    "parse_error_type",
]].head(5))


saved csv: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009/llm_answer_review.csv
saved html: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009/llm_answer_review.html


,question_id,question,ground_truth,parsed_answer,final_answer,valid_json,recovered_answer,parse_error_type
0,Q005,그랜드코리아레저(주)의 그룹웨어 구축 사업에서 목표로 하는 시스템 개선의 대상 범위...,"사업 범위는 그룹웨어 시스템 구축, 기록물관리 시스템 구축, 업무용 메신저 시스템 ...",시스템 개선의 대상 범위는 다음과 같다: 임직원의 업무 처리를 위한 통합창구 기능을...,,False,,
1,Q008,한국수자원공사의 '용인 첨단 시스템반도체 국가산단 용수공급사업'과 한국수출입은행의 ...,"한국수자원공사 용역 예산(2,392,940,000원)과 수출입은행 용역 예산(1,2...","두 사업의 예산 차액은 100,000,000원입니다.",,False,,
2,Q021,사단법인아시아물위원회사무국에서 발주한 '우즈벡-키르기즈스탄 기후변화대응 스마트 관개...,"해당 사업의 예산은 5,031,000,000원입니다.",사단법인아시아물위원회사무국에서 발주한 '우즈벡-키르기즈스탄 기후변화대응 스마트 관개...,,False,,
3,Q039,그렌드코리아레져(쥬)가 이번 연도에 츄진하는 2024년도 구룹웨에 씨스탬 구쭉 용역...,그랜드코리아레저(주)가 추진을 기획하고 있는 '2024년도 GKL 그룹웨어 시스템 ...,2024년도 GKL 그룹웨어 시스템 구축 용역의 전체 에산은 약 80억원입니다.,2024년도 GKL 그룹웨어 시스템 구축 용역의 전체 에산은 약 80억원입니다.,True,,
4,Q041,한국가스공사의 차세대 통합정보시스템(ERP) 구축 사업의 신규 도입 S/W 중 전사...,"네, 해당 사업의 과업내용 중 주요 사업내용으로 본사 전사 포털 및 업무 포털의 신...",전사 포털 및 사용자 맞춤형 업무 포털이 신규 도입될 S/W 분야입니다.,전사 포털 및 사용자 맞춤형 업무 포털이 신규 도입될 S/W 분야입니다.,True,,


## 17. ???? ?? ?? ??

review50?? ??? ??? ??? QID? ???? ?? generation ??? ???? ????? ?????.
? ??? RAGAS?? ?? ?? deterministic ?????.


In [25]:
from collections import Counter

failure_counter = Counter()
answer_status_counter = Counter()
answer_type_counter = Counter()
for record in records:
    failure_counter.update(record.get("_failure_tags", []) or [])
    answer_status_counter[str(record.get("answer_status", ""))] += 1
    answer_type_counter[str(record.get("answer_type", ""))] += 1

print("total:", len(records))
print("empty answers:", sum(not str(r.get("answer", "")).strip() for r in records))
print("valid json:", sum(bool(r.get("_valid_json")) for r in records), "/", len(records))
print("answer_status:", dict(answer_status_counter))
print("answer_type:", dict(answer_type_counter))
print("failure_tags:", dict(failure_counter))

summary_path = OUTPUT_DIR / "metrics_summary.json"
if summary_path.exists():
    print("metrics_summary:")
    print(json.dumps(json.loads(summary_path.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))


total: 50
empty answers: 26
valid json: 24 / 50
answer_status: {'': 50}
answer_type: {'summary': 18, 'budget': 11, 'business_type': 3, 'general': 4, 'unknown': 4, 'multi_doc_comparison': 10}
failure_tags: {'llm_invalid_json': 26, 'insufficient_evidence': 7, 'llm_hallucination_risk': 6, 'incomplete_multi_doc': 7}
metrics_summary:
{
  "total_questions": 50,
  "total": 50,
  "valid_json_count": 24,
  "valid_json_rate": 0.48,
  "citation_checked_count": 50,
  "citation_valid_rate": 0.86,
  "numeric_grounded_checked_count": 50,
  "numeric_grounded_rate": 0.88,
  "answerable_count": 19,
  "answerable_rate": 0.38,
  "generation_ms_avg": 26688.882862740018,
  "failure_tag_counts": {
    "llm_invalid_json": 26,
    "insufficient_evidence": 7,
    "llm_hallucination_risk": 6,
    "incomplete_multi_doc": 7
  }
}


In [26]:
KEY_QIDS = ["Q008", "Q057", "Q100", "Q128", "Q147", "Q166", "Q186", "Q290", "Q326", "Q386", "Q486"]

rows = []
for qid in KEY_QIDS:
    record = next((r for r in records if r.get("question_id") == qid), None)
    if not record:
        rows.append({"question_id": qid, "status": "not_in_sample"})
        continue
    rows.append({
        "question_id": qid,
        "answer_type": record.get("answer_type"),
        "answer_status": record.get("answer_status"),
        "is_answerable": record.get("is_answerable"),
        "answer": str(record.get("answer", ""))[:260],
        "computed_values": json.dumps(record.get("computed_values", {}), ensure_ascii=False),
        "failure_tags": json.dumps(record.get("_failure_tags", []), ensure_ascii=False),
        "missing_info": json.dumps(record.get("missing_info", []), ensure_ascii=False),
        "source_files": json.dumps(record.get("source_files", []), ensure_ascii=False),
    })

key_case_df = pd.DataFrame(rows)
display(key_case_df)
key_case_df.to_csv(OUTPUT_DIR / "generation_key_case_regression.csv", index=False, encoding="utf-8-sig")
print("saved:", OUTPUT_DIR / "generation_key_case_regression.csv")


,question_id,answer_type,answer_status,is_answerable,answer,computed_values,failure_tags,missing_info,source_files
0,Q008,summary,None,False,,{},"[""llm_invalid_json""]","[""valid_json""]","[""한국수출입은행_(긴급) 모잠비크 마푸토 지능형교통시스템(ITS) 구축사업.hwp..."
1,Q057,unknown,None,False,"Context에서 주관적인 정보만 제공되므로, 해당 조건에 대한 명확한 답변을 제공...",{},[],[],"[""한국수자원공사_용인 첨단 시스템반도체 국가산단 용수공급사업 타당성.hwp""]"
2,Q100,budget,None,False,,{},"[""llm_invalid_json""]","[""valid_json""]","[""나노종합기술원_스마트 팹 서비스 활용체계 구축관련 설비온라인 시스.hwp"", ""..."
3,Q128,summary,None,False,,{},"[""llm_invalid_json""]","[""valid_json""]","[""인천광역시_인천일자리플랫폼 정보시스템 구축 ISP 수립용역.hwp"", ""한국수자..."
4,Q147,summary,None,False,,{},"[""llm_invalid_json""]","[""valid_json""]","[""인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp"",..."
5,Q166,budget,None,True,"초기 계약 선수금으로 전체 예산의 25%가 선지급되므로, 996,356,000원의 ...",{},"[""llm_hallucination_risk""]",[],"[""인천공항운영서비스(주)_인천공항운영서비스㈜ 차세대 ERP시스템 구축 .hwp"",..."
6,Q186,budget,None,True,"신규 코딩비에 배정되는 금액은 75,900,000원입니다.",{},"[""llm_hallucination_risk""]",[],"[""한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp""]"
7,Q290,budget,None,True,"390,060,000원",{},"[""llm_hallucination_risk""]",[],"[""한국수자원공사_수도사업장 통합 사고분석솔루션 시범구축 용역.hwp"", ""한국수자..."
8,Q326,budget,None,True,최종적으로 순수 개발사가 소프트웨어 개발 인건비로 남길 수 있는 자금 여력의 액수는...,{},"[""insufficient_evidence"", ""llm_hallucination_r...",[],"[""광주과학기술원_실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업.hwp""]"
9,Q386,summary,None,False,,{},"[""llm_invalid_json""]","[""valid_json""]","[""광주과학기술원_실시간통합연구비관리시스템(RCMS) 연계 모듈 변경 사업.hwp""]"


saved: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009/generation_key_case_regression.csv


In [27]:
def _find_record(qid):
    return next((r for r in records if r.get("question_id") == qid), None)

checks = {
    "Q008_wrong_495m_blocked": lambda r: (r is None) or ("495,000,000" not in str(r.get("answer", "")) and "495000000" not in json.dumps(r.get("final_values", {}), ensure_ascii=False)),
    "Q057_not_budget": lambda r: (r is None) or (r.get("answer_type") != "budget"),
    "Q100_has_summary_hint": lambda r: (r is None) or ("??" in str(r.get("answer", "")) or "??" in str(r.get("answer", "")) or "R&D" in str(r.get("answer", "")) or "??" in str(r.get("answer", ""))),
    "Q326_expected_value": lambda r: (r is None) or ("30,056,400" in str(r.get("answer", "")) or "30056400" in json.dumps(r.get("final_values", {}), ensure_ascii=False)),
    "Q386_expected_value": lambda r: (r is None) or ("4,356,000" in str(r.get("answer", "")) or "4356000" in json.dumps(r.get("final_values", {}), ensure_ascii=False)),
    "Q486_expected_value": lambda r: (r is None) or ("145,333" in str(r.get("answer", "")) or "145333" in json.dumps(r.get("final_values", {}), ensure_ascii=False)),
}

check_rows = []
for name, fn in checks.items():
    qid = name.split("_")[0]
    record = _find_record(qid)
    check_rows.append({"check": name, "passed": bool(fn(record)), "question_in_sample": record is not None})

check_df = pd.DataFrame(check_rows)
display(check_df)
check_df.to_csv(OUTPUT_DIR / "generation_regression_checks.csv", index=False, encoding="utf-8-sig")
print("saved:", OUTPUT_DIR / "generation_regression_checks.csv")


,check,passed,question_in_sample
0,Q008_wrong_495m_blocked,True,True
1,Q057_not_budget,True,True
2,Q100_has_summary_hint,False,True
3,Q326_expected_value,False,True
4,Q386_expected_value,False,True
5,Q486_expected_value,False,True


saved: /content/drive/MyDrive/DB_RAG_Codeit_Project/outputs/generation_p4_hwpx_125/generation_review50_J5_hybrid_rrf_rerank_20260525_033009/generation_regression_checks.csv
